## HateXplain - 3 OBJECTIVES

### Multi Task Learning

* **Dataset:** HateXplain
* **Task :** Classification
* **Target 1:** Hate speech
* **Target 2:** Hate speech against woman and homosexual
* **Target 3:** Hate speech against ethinicity

### Methods

* **MOLA**
* **Random Weights**

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from transformers import AutoModel
from collections import Counter
from tqdm import tqdm
from datasets import load_dataset

/home/lineccsa/mestrado/moo_research/moo_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from ml_moo import moo, get_objectives, MooScalarization
from ml_moo.analysis.metrics import compute_hypervolume_progress
from ml_moo.analysis.visualization import plot_pareto_3d, plot_multiple_hypervolumes, plot_hypervolume
from ml_moo.scalarization.three_objs_hate_speech import HateSpeechScalarization

In [4]:
# Definir seed para reprodutibilidade
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

## Dataset and Model

In [5]:
# Carregar o dataset HateXplain
dataset = load_dataset("Hate-speech-CNERG/hatexplain", trust_remote_code=True)


In [6]:
# model_name = "tum-nlp/bert-hateXplain"
#"hate-bert, uncased bert"
model_name="bert-base-uncased"

In [7]:
print("GPU disponível:", torch.cuda.is_available())

GPU disponível: True


In [8]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [9]:
class MultiTaskHateSpeechDataset(Dataset):
    def __init__(self, dataset, tokenizer_name="bert-base-uncased", max_length=128):
        self.dataset = dataset
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_length = max_length
        self.bert_model = AutoModel.from_pretrained(tokenizer_name)
        self.bert_model.eval()

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        data = self.dataset[idx]

        # Converter lista de tokens em texto
        text = " ".join(data["post_tokens"])

        # Encontrar a moda dos rótulos (classificação de discurso de ódio)
        labels = data["annotators"]["label"]
        label_counts = Counter(labels)
        majority_label = label_counts.most_common(1)[0][0]  # Pega a moda
      
        label_hate = 0 if majority_label == 1 else 1

        # Criar rótulo para discurso de ódio contra gênero (exemplo: "women")
        target_groups = data["annotators"]["target"]
        #label_target = 1 if len(target_groups) > 0 else 0
        count_gender = 0
        count_race = 0

        for i in range(len(target_groups)):
            #print(target_groups[i])
            if ('Women' in target_groups[i] or 'Homosexual' in target_groups[i]):
                count_gender += 1
            # etnias
            if ('Indigenous' in target_groups[i] or 'African' in target_groups[i] or 'Asian' in target_groups[i] or 'Jewish' in target_groups):
                count_race += 1

        gender_hate = 1 if ((count_gender > 0) and (label_hate == 1)) else 0  # 1 se houver ataque a mulheres, 0 caso contrário
        race_hate = 1 if ((count_race > 0) and (label_hate == 1)) else 0

        # Tokenizar texto
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(label_hate, dtype=torch.long),
            "gender_hate": torch.tensor(gender_hate, dtype=torch.long),
            "race_hate": torch.tensor(race_hate, dtype=torch.long)
            #"label_target": label_target
            #"embedding": embedding.squeeze(0)
        }

In [10]:
# Criar os datasets para treino, validação e teste
train_dataset = MultiTaskHateSpeechDataset(dataset["train"])
val_dataset = MultiTaskHateSpeechDataset(dataset["train"])
test_dataset = MultiTaskHateSpeechDataset(dataset["train"])

# DataLoaders
batch_size = 8
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(val_dataset, batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

In [11]:
opt_params = {
    'node_time_limit': 2,
    'target_size': 50,
    'target_gap': 0,
    'node_gap': 0.05,
    'norm': False
}
results = {}

## MOLA

ml_moo = 360 min - 300 iter - 25 solucoes

50 iter = 98 min 16 s

In [ ]:
method = 'mola'
w_scalar = HateSpeechScalarization(train_dataloader=train_dataloader, device=device, model_name=model_name)
moopt = moo(w_scalar).mo_optimization(method, **opt_params)
objs = get_objectives(moopt)
#hypervolume_values = compute_hypervolume_progress(objs)
hypervolume_values = moopt.get_hypervolumes()
results[method] = {
                "moopt": moopt,
                "objectives": objs,
                "models": moopt.get_models() if hasattr(moopt, "get_models") else None,
                "hypervolume": hypervolume_values
            }

hidden size 768


11:51:48 | DEBUG | ml_moo | Filtered parameters for Mola: {'node_time_limit': 2, 'target_size': 50, 'target_gap': 0, 'node_gap': 0.05, 'norm': False}
11:51:48 | DEBUG | ml_moo | Finding 1th individual minimum


device: cuda:0
available: True


KeyboardInterrupt: 

: 

In [ ]:
target = ["General", "Women", "Homosexual"]

In [ ]:
title="Conflict between losses in training"
plot_pareto_3d(objectives=objs, 
               labels=target, 
               title=title,
               #point = equal.objs,
               color="#FF5C8D"
               )

## Random Weight

300 iter = 585 min 11 s 
50 iter = 92 min 53 s

In [ ]:
method = 'random_weight'
w_scalar = HateSpeechScalarization(train_dataloader=train_dataloader, device=device, model_name=model_name)
moopt = moo(w_scalar).mo_optimization(method, **opt_params)
objs = get_objectives(moopt)
hypervolume_values = compute_hypervolume_progress(objs)
results[method] = {
                "moopt": moopt,
                "objectives": objs,
                "models": moopt.get_models() if hasattr(moopt, "get_models") else None,
                "hypervolume": hypervolume_values
            }

device: cuda:0
available: True


Epoch 1/3 - Training: 100%|██████████| 1923/1923 [00:40<00:00, 47.49it/s]


Epoch 1 - Average training loss: 0.4626


Epoch 2/3 - Training: 100%|██████████| 1923/1923 [00:39<00:00, 48.18it/s]


Epoch 2 - Average training loss: 0.4095


...

Epoch 2/3 - Training: 100%|██████████| 1923/1923 [00:33<00:00, 57.69it/s]


Epoch 2 - Average training loss: 0.3691


Epoch 3/3 - Training: 100%|██████████| 1923/1923 [00:32<00:00, 59.82it/s]


Epoch 3 - Average training loss: 0.3253


Evaluating: 100%|██████████| 1923/1923 [00:12<00:00, 159.92it/s]


## Comparison between MOO methods

In [ ]:
from ml_moo.tests.experiments.run import run_experiments
from ml_moo.analysis.moo_analyzer import Analyzer

In [ ]:
pareto_dict = {
        method: res["objectives"]
        for method, res in results.items()
        if "objectives" in res
    }

hypervolumes_dict = {
        method: res["hypervolume"]
        for method, res in results.items()
        if "hypervolume" in res
    }

Analyzer.run_multiple(pareto_dict = pareto_dict,
                      hypervolumes_values = hypervolumes_dict,
                      show_pareto = True, 
                      show_hypervolume = True,
                      subplots = True,
                      subplots_hv = True,
                      )